# 15.15 Union-Find, Tries and Bit Manipulation

**Prerequisites:** 15.9 Graphs, 15.6 Hashing, 15.7 Trees  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- **Union-Find** - connected components as edges arrive, in nearly O(1)
- Path compression and union by rank, **measured**
- Kruskal's minimum spanning tree, built on it
- **Tries** - prefix trees for autocomplete and prefix queries
- 🔴 Trie vs hash map: what a trie buys, and what it costs
- **Bit manipulation** - the operators, and the tricks worth memorising
- 🔴 XOR's three properties, and the problems they solve in one line
- **Bitmasks** as sets - subsets, and bitmask DP
- Interview questions, worked

---

---

# Union-Find (Disjoint Set Union)

**15.9** found connected components with BFS: O(V + E), but it must re-run from scratch whenever an edge is added.

Union-Find answers two questions **incrementally**, as edges arrive:

| Operation | Question |
|---|---|
| `find(x)` | which group is `x` in? |
| `union(x, y)` | merge the groups containing `x` and `y` |

```
   each group is a TREE; the root is the group's identity

        1          4            find(3) walks up to 1
       / \          \           find(5) walks up to 4
      2   3          5          different roots -> different groups

   union(3, 5): point one root at the other

        1
       /|\
      2 3 4
          \
           5
```

🔴 **Naively this degenerates** exactly like the BST in **15.7**: union in the wrong order and you build a chain, making `find` O(n).

Two optimisations fix it, and together they are remarkable:

| Optimisation | Idea |
|---|---|
| **Path compression** | after `find`, point every node on the path directly at the root |
| **Union by rank/size** | always attach the *smaller* tree under the larger |

> With both, the amortised cost is **O(α(n))**, where α is the inverse Ackermann function. For any n that fits in the observable universe, **α(n) < 5**. It is not constant, but nothing you will ever run can tell the difference.

In [ ]:
class UnionFind:
    """Disjoint sets with path compression and union by size."""

    def __init__(self, items):
        self.parent = {item: item for item in items}     # each is its own root
        self.size = {item: 1 for item in items}
        self.groups = len(self.parent)
        self.steps = 0                                   # instrumentation

    def find(self, item):
        """Root of item's group, compressing the path on the way."""
        root = item
        while self.parent[root] != root:
            self.steps += 1
            root = self.parent[root]
        # PATH COMPRESSION: re-point everything we walked past
        while self.parent[item] != root:
            self.parent[item], item = root, self.parent[item]
        return root

    def union(self, a, b):
        """Merge two groups. Returns False if they were already joined."""
        root_a, root_b = self.find(a), self.find(b)
        if root_a == root_b:
            return False
        # UNION BY SIZE: hang the smaller tree under the larger
        if self.size[root_a] < self.size[root_b]:
            root_a, root_b = root_b, root_a
        self.parent[root_b] = root_a
        self.size[root_a] += self.size[root_b]
        self.groups -= 1
        return True

    def connected(self, a, b):
        return self.find(a) == self.find(b)

    def component_of(self, item):
        root = self.find(item)
        return sorted(x for x in self.parent if self.find(x) == root)


sets = UnionFind("abcdef")
print("start:", sets.groups, "groups\n")

for a, b in (("a", "b"), ("c", "d"), ("b", "c"), ("a", "d"), ("e", "f")):
    merged = sets.union(a, b)
    note = "merged" if merged else "already connected"
    print(f"  union({a}, {b}): {note:<20} groups now {sets.groups}")

print("\n  a and d connected:", sets.connected("a", "d"))
print("  a and e connected:", sets.connected("a", "e"))
print("  group containing a:", sets.component_of("a"))
print("  group containing e:", sets.component_of("e"))
print("\n  🔴 union('a','d') returned False - they were ALREADY in the same")
print("     group. That return value is a CYCLE DETECTOR, and it is what")
print("     Kruskal's algorithm uses below.")

In [ ]:
# What do the two optimisations actually buy? Measure the naive version.
class NaiveUnionFind:
    """No path compression, no union by size. The pathological case."""

    def __init__(self, items):
        self.parent = {item: item for item in items}
        self.steps = 0

    def find(self, item):
        while self.parent[item] != item:
            self.steps += 1
            item = self.parent[item]
        return item

    def union(self, a, b):
        root_a, root_b = self.find(a), self.find(b)
        if root_a != root_b:
            self.parent[root_b] = root_a      # always the same direction


print(f"{'n':>8}{'naive steps':>15}{'optimised steps':>18}{'ratio':>10}")
print("-" * 52)
for n in (500, 1_000, 2_000, 4_000):
    items = list(range(n))

    # the worst order for the naive version: build one long chain
    naive = NaiveUnionFind(items)
    for i in range(n - 1):
        naive.union(i + 1, i)
    for i in items:
        naive.find(i)

    smart = UnionFind(items)
    for i in range(n - 1):
        smart.union(i + 1, i)
    for i in items:
        smart.find(i)

    print(f"{n:>8}{naive.steps:>15,}{smart.steps:>18,}"
          f"{naive.steps / max(smart.steps, 1):>9,.0f}x")

print("\n  The naive version builds a chain and every find walks it: O(n^2)")
print("  overall. With compression and union by size the total is nearly")
print("  linear - the steps barely grow.")
print("\n  🔴 Same data structure, same operations. The difference is two")
print("     small rules about WHICH way to attach and what to do on the")
print("     way back up.")

### Kruskal's minimum spanning tree

*Connect every vertex at the lowest total edge cost.* Cabling a network, laying pipe, clustering.

**The algorithm is three lines**, and it is greedy (**15.14**):

```
   sort the edges by weight
   for each edge, cheapest first:
       if it joins two DIFFERENT components:  take it
```

Union-Find is what makes the middle test O(α(n)). And `union` returning `False` — the "already connected" signal — is exactly the cycle check.

| | |
|---|---|
| Time | O(E log E) — dominated by the sort |
| Result | V−1 edges, no cycles, minimum total weight |

> Greedy is **provably optimal** here (the cut property), unlike coin change or 0/1 knapsack (**15.13**). It is a good example to have ready when asked *"when does greedy work?"*

In [ ]:
def kruskal(vertices, edges):
    """Minimum spanning tree. edges are (weight, a, b). O(E log E)."""
    sets = UnionFind(vertices)
    chosen = []
    total = 0
    for weight, a, b in sorted(edges):          # greedy: cheapest first
        if sets.union(a, b):                    # False means it would cycle
            chosen.append((weight, a, b))
            total += weight
            if len(chosen) == len(vertices) - 1:
                break                           # a tree is complete
    return chosen, total, sets.groups


vertices = "ABCDEF"
edges = [
    (4, "A", "B"), (3, "B", "C"), (2, "A", "C"), (5, "C", "D"),
    (7, "B", "D"), (6, "D", "E"), (1, "E", "F"), (8, "C", "F"),
]

chosen, total, groups = kruskal(vertices, edges)
print(f"{len(edges)} edges, {len(vertices)} vertices\n")
print("minimum spanning tree:")
for weight, a, b in chosen:
    print(f"  {a}-{b}  weight {weight}")
print(f"\n  total weight : {total}")
print(f"  edges chosen : {len(chosen)} (V-1 = {len(vertices) - 1})")
print(f"  components   : {groups} (1 means fully connected)")

# a graph that cannot be spanned
disconnected = [(1, "A", "B"), (2, "C", "D")]
chosen, total, groups = kruskal("ABCD", disconnected)
print(f"\n  on a disconnected graph: {len(chosen)} edges, {groups} components")
print("  ^ fewer than V-1 edges means no spanning tree exists - the graph")
    
print("    is in pieces. Worth checking rather than assuming.")

---

# Tries (prefix trees)

A tree where **the path spells the word**.

```
        (root)
        /    \
       c      d
       |      |
       a      o
      / \     |
     r   t*   g*        * marks the end of a word
     |
     t*

   contains: cart, cat, dog
```

🔴 **The end-of-word marker is essential.** Without it you cannot distinguish a stored word from a mere prefix — `cat` is a word, `ca` is not, and both are paths in the tree.

| Operation | Trie | Hash set (**15.6**) |
|---|---|---|
| Insert / search a word | O(L) — length of the word | **O(L)** to hash, then O(1) |
| **All words with a prefix** | ✅ **O(L + results)** | 🔴 O(n) — scan everything |
| Memory | 🔴 higher — a node per character | lower |
| Sorted traversal | ✅ free, in alphabetical order | 🔴 needs a sort |

> **The one thing a trie does that a hash map cannot: prefix queries.** Autocomplete, spell-check suggestions, IP routing tables, and T9 predictive text are all tries.

If you never need prefixes, **use a `set`** — it is faster and smaller.

In [ ]:
class TrieNode:
    __slots__ = ("children", "is_word")

    def __init__(self):
        self.children = {}                # character -> TrieNode
        self.is_word = False              # 🔴 without this, 'ca' looks like a word


class Trie:
    def __init__(self, words=()):
        self.root = TrieNode()
        self.count = 0
        for word in words:
            self.insert(word)

    def insert(self, word):
        node = self.root
        for char in word:
            node = node.children.setdefault(char, TrieNode())
        if not node.is_word:
            node.is_word = True
            self.count += 1

    def _walk(self, prefix):
        """Node at the end of prefix, or None."""
        node = self.root
        for char in prefix:
            if char not in node.children:
                return None
            node = node.children[char]
        return node

    def contains(self, word):
        node = self._walk(word)
        return node is not None and node.is_word

    def starts_with(self, prefix):
        return self._walk(prefix) is not None

    def with_prefix(self, prefix):
        """Every stored word beginning with prefix - in alphabetical order."""
        node = self._walk(prefix)
        if node is None:
            return []
        found = []

        def collect(current, so_far):
            if current.is_word:
                found.append(prefix + so_far)
            for char in sorted(current.children):      # sorted -> alphabetical
                collect(current.children[char], so_far + char)

        collect(node, "")
        return found


words = ["cat", "cart", "car", "dog", "do", "door", "cats"]
trie = Trie(words)
print(f"stored {trie.count} words\n")

for probe in ("cat", "ca", "do", "doo", "dogs"):
    print(f"  {probe!r:<8} is a word: {str(trie.contains(probe)):<6} "
          f"is a prefix: {trie.starts_with(probe)}")

print("\n  🔴 'ca' is a prefix but NOT a word. That distinction is exactly")
print("     what is_word records - a hash set could not tell you the first")
print("     part at all.\n")

print("autocomplete:")
for prefix in ("ca", "do", "x", ""):
    print(f"  {prefix!r:<6} -> {trie.with_prefix(prefix)}")
print("\n  Results come out alphabetically for free - the trie is already")
print("  ordered by character (15.7's in-order idea, generalised).")

In [ ]:
# The trade, measured: prefix search against a set and a sorted list.
import bisect
import random
import sys
import time

rng = random.Random(15)
alphabet = "abcdefgh"
vocabulary = sorted({"".join(rng.choices(alphabet, k=rng.randint(3, 8)))
                     for _ in range(20_000)})

trie = Trie(vocabulary)
as_set = set(vocabulary)
PREFIX = "abc"

started = time.perf_counter()
for _ in range(50):
    from_trie = trie.with_prefix(PREFIX)
trie_time = time.perf_counter() - started

started = time.perf_counter()
for _ in range(50):
    from_set = [w for w in as_set if w.startswith(PREFIX)]
set_time = time.perf_counter() - started

started = time.perf_counter()
for _ in range(50):
    left = bisect.bisect_left(vocabulary, PREFIX)
    from_sorted = []
    while left < len(vocabulary) and vocabulary[left].startswith(PREFIX):
        from_sorted.append(vocabulary[left])
        left += 1
sorted_time = time.perf_counter() - started

print(f"{len(vocabulary):,} words, 50 prefix queries for {PREFIX!r} "
      f"({len(from_trie)} matches)\n")
print(f"  trie            {trie_time * 1000:8.1f} ms   O(L + results)")
print(f"  scan a set      {set_time * 1000:8.1f} ms   O(n)")
print(f"  sorted+bisect   {sorted_time * 1000:8.1f} ms   O(log n + results)")
print(f"\n  all agree: {sorted(from_trie) == sorted(from_set) == sorted(from_sorted)}")

print("\n🔴 The honest conclusion: for a STATIC word list, a sorted list plus")
print("   bisect (15.11) matches a trie and uses far less memory. The trie")
print("   wins when the set changes constantly, since insertion is O(L)")
print("   rather than O(n) (15.2).")
print(f"\n  memory: set {sys.getsizeof(as_set):,} bytes for the container alone;")
print("  the trie allocates one object per character node.")

---

# Bit manipulation

| Operator | Name | Effect |
|---|---|---|
| `a & b` | AND | 1 where **both** are 1 |
| `a \| b` | OR | 1 where **either** is 1 |
| `a ^ b` | XOR | 1 where they **differ** |
| `~a` | NOT | flips every bit (in Python, `~a == -a-1`) |
| `a << n` | left shift | multiply by 2ⁿ |
| `a >> n` | right shift | integer-divide by 2ⁿ |

### The four standard manipulations

```
    check bit i:   (x >> i) & 1        or  x & (1 << i)
    set bit i:     x |  (1 << i)
    clear bit i:   x & ~(1 << i)
    toggle bit i:  x ^  (1 << i)
```

🔴 **Python integers are arbitrary precision and conceptually infinite two's complement.** So `~5 == -6`, not `250` — there is no fixed width to wrap around. Masking with `& 0xFF` is how you emulate a byte.

In [ ]:
def show_bits(value, width=8):
    return format(value & (2 ** width - 1), f"0{width}b")


a, b = 0b1100, 0b1010
print(f"  a       = {show_bits(a)}  = {a}")
print(f"  b       = {show_bits(b)}  = {b}")
print(f"  a & b   = {show_bits(a & b)}  = {a & b}")
print(f"  a | b   = {show_bits(a | b)}  = {a | b}")
print(f"  a ^ b   = {show_bits(a ^ b)}  = {a ^ b}")
print(f"  a << 1  = {show_bits(a << 1)}  = {a << 1}   (x2)")
print(f"  a >> 1  = {show_bits(a >> 1)}  = {a >> 1}   (//2)")

print(f"\n  🔴 ~a = {~a}, not {show_bits(~a)} as an unsigned byte.")
print(f"     Python has no fixed width: ~x == -x-1. Mask to emulate one:")
print(f"     ~a & 0xFF = {~a & 0xFF} = {show_bits(~a & 0xFF)}")

print("\nthe four manipulations on x = 0b1010 (bit 2 is 0, bit 1 is 1):")
x = 0b1010
for i in (1, 2):
    print(f"  bit {i}: check={x >> i & 1}  "
          f"set={show_bits(x | (1 << i))}  "
          f"clear={show_bits(x & ~(1 << i))}  "
          f"toggle={show_bits(x ^ (1 << i))}")

## 🔴 XOR - three properties, many one-liners

```
    1.  x ^ x = 0            a value cancels itself
    2.  x ^ 0 = x            zero is the identity
    3.  XOR is commutative and associative - order does not matter
```

Together these mean: **XOR everything together and the pairs vanish.**

| Problem | Solution |
|---|---|
| One number appears once, all others twice — find it | XOR the whole array |
| Find the missing number in `0..n` | XOR the array with `0..n` |
| Swap two variables without a temporary | `a^=b; b^=a; a^=b` |
| Detect whether two numbers differ | `a ^ b != 0` |

The first is the classic: **O(n) time, O(1) space**, where a hash map (**15.6**) would need O(n) memory and sorting would need O(n log n).

🔴 The XOR swap is a **party trick, not advice** — it is slower than tuple unpacking in Python, and it silently zeroes the value if both names refer to the same variable.

In [ ]:
import functools
import operator


def single_number(data):
    """Every value appears twice except one. O(n) time, O(1) space."""
    return functools.reduce(operator.xor, data, 0)


def missing_number(data, n):
    """One number missing from 0..n. XOR everything - pairs cancel."""
    result = 0
    for i in range(n + 1):
        result ^= i
    for value in data:
        result ^= value
    return result


print("  single_number([4,1,2,1,2])  =", single_number([4, 1, 2, 1, 2]))
print("  single_number([7])          =", single_number([7]))
print("  missing_number([0,1,3], 3)  =", missing_number([0, 1, 3], 3))
print("  missing_number([1,2,3], 3)  =", missing_number([1, 2, 3], 3))

rng = random.Random(15)
ok = True
for _ in range(300):
    n = rng.randint(1, 30)
    pairs = [rng.randint(0, 100) for _ in range(n)]
    lonely = rng.randint(200, 300)
    data = pairs + pairs + [lonely]
    rng.shuffle(data)
    if single_number(data) != lonely:
        ok = False
print(f"\n  single_number correct on 300 random cases: {ok}")

# 🔴 the XOR swap, and why not to use it
a, b = 5, 9
a ^= b
b ^= a
a ^= b
print(f"\n  XOR swap: a={a}, b={b}")

values = [3, 7]
i = j = 0                                   # the same index!
values[i] ^= values[j]
values[j] ^= values[i]
values[i] ^= values[j]
print(f"  🔴 XOR-swapping an element with ITSELF: {values}")
print("     The value became 0. Tuple unpacking `a, b = b, a` is faster,")
print("     clearer, and has no such failure mode.")

## Counting bits, and powers of two

### Brian Kernighan's trick

```
    x & (x - 1)    clears the LOWEST set bit

    x     = 1011000
    x - 1 = 1010111       borrowing flips the lowest 1 and everything below
    x&x-1 = 1010000       the lowest 1 is gone
```

So counting set bits takes **one iteration per set bit**, not per bit — much better on sparse values.

> ### Version note - `int.bit_count()`, new in 3.10
> Python now has it built in, in C: `(0b1011000).bit_count() == 3`. Use it. Know Kernighan's trick for interviews, where implementing it is the question.

### Powers of two

```
    x > 0 and x & (x - 1) == 0     x is a power of two
```

A power of two has **exactly one** set bit, so clearing the lowest leaves zero. The `x > 0` guard matters: `0 & -1 == 0` would otherwise report 0 as a power of two.

In [ ]:
def count_bits_naive(x):
    count = 0
    while x:
        count += x & 1
        x >>= 1                       # one iteration per BIT
    return count


def count_bits_kernighan(x):
    count = 0
    while x:
        x &= x - 1                    # one iteration per SET bit
        count += 1
    return count


def is_power_of_two(x):
    return x > 0 and x & (x - 1) == 0


print(f"{'value':>12}{'binary':>20}{'naive':>8}{'kernighan':>11}{'bit_count':>11}")
print("-" * 64)
for value in (0, 1, 7, 8, 255, 1024, 0b1011000):
    print(f"{value:>12}{format(value, 'b'):>20}"
          f"{count_bits_naive(value):>8}{count_bits_kernighan(value):>11}"
          f"{value.bit_count():>11}")

print("\n  Kernighan wins on sparse values: 1024 is one set bit in 11 bits,")
print("  so it loops once instead of eleven times.")

big = 1 << 200                        # a single set bit, 201 bits wide
print(f"\n  for 2^200: naive loops 201 times, Kernighan loops "
      f"{count_bits_kernighan(big)} time")
print(f"  and int.bit_count() is C: {big.bit_count()}")

print("\npowers of two:")
for value in (0, 1, 2, 3, 16, 18, 1024):
    print(f"  {value:>5} -> {is_power_of_two(value)}")
print("  🔴 the x > 0 guard is what keeps 0 out - 0 & -1 == 0 otherwise")
print("     reports it as a power of two.")

## Bitmasks as sets

An integer's bits **are** a subset indicator: bit `i` set means item `i` is included.

```
   items:  [a, b, c]

   bit 0 -> items[0] = a,  bit 1 -> items[1] = b,  bit 2 -> items[2] = c

   0b000 = {}          0b100 = {c}
   0b001 = {a}         0b101 = {a, c}
   0b010 = {b}         0b110 = {b, c}
   0b011 = {a, b}      0b111 = {a, b, c}
```

So **counting from 0 to 2ⁿ−1 enumerates every subset** — an iterative alternative to the backtracking version in **15.12**.

| Set operation | Bitmask |
|---|---|
| union | `a \| b` |
| intersection | `a & b` |
| difference | `a & ~b` |
| add item i | `a \| (1 << i)` |
| remove item i | `a & ~(1 << i)` |
| is i present | `a & (1 << i)` |
| size | `a.bit_count()` |

> **Bitmask DP** (**15.13**) uses this to make the *set of visited items* a DP state — which is how the travelling salesman problem is solved in O(2ⁿ·n) instead of O(n!).

🔴 Only practical for **n ≤ ~20**: 2²⁰ is a million states, 2³⁰ is a billion.

In [ ]:
def subsets_bitmask(items):
    """Every subset, by counting from 0 to 2^n - 1."""
    n = len(items)
    out = []
    for mask in range(1 << n):                     # 2^n masks
        out.append([items[i] for i in range(n) if mask & (1 << i)])
    return out


items = ["a", "b", "c"]
print(f"{'mask':>6}{'binary':>9}   subset")
print("-" * 30)
for mask in range(1 << len(items)):
    subset = [items[i] for i in range(len(items)) if mask & (1 << i)]
    print(f"{mask:>6}{format(mask, '03b'):>9}   {subset}")

print(f"\n  {len(subsets_bitmask(items))} subsets = 2^{len(items)}")

# set operations
A_SET = 0b0101          # items 0 and 2
B_SET = 0b0110          # items 1 and 2
print(f"\n  A        = {format(A_SET, '04b')}")
print(f"  B        = {format(B_SET, '04b')}")
print(f"  union    = {format(A_SET | B_SET, '04b')}")
print(f"  intersect= {format(A_SET & B_SET, '04b')}")
print(f"  A - B    = {format(A_SET & ~B_SET, '04b')}")
print(f"  |A|      = {A_SET.bit_count()}")

# a real bitmask DP: travelling salesman
def tsp(distances):
    """Shortest tour visiting every city once. O(2^n * n^2) instead of O(n!)."""
    n = len(distances)
    INF = float("inf")
    # best[mask][last] = cheapest way to visit `mask`, ending at `last`
    best = [[INF] * n for _ in range(1 << n)]
    best[1][0] = 0                                  # start at city 0

    for mask in range(1 << n):
        for last in range(n):
            if best[mask][last] == INF or not mask & (1 << last):
                continue
            for nxt in range(n):
                if mask & (1 << nxt):
                    continue                        # already visited
                new_mask = mask | (1 << nxt)
                candidate = best[mask][last] + distances[last][nxt]
                if candidate < best[new_mask][nxt]:
                    best[new_mask][nxt] = candidate

    full = (1 << n) - 1
    return min(best[full][last] + distances[last][0] for last in range(1, n))


distances = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0],
]
print(f"\n  TSP over 4 cities: shortest tour = {tsp(distances)}")

import itertools

brute = min(
    sum(distances[tour[i]][tour[i + 1]] for i in range(len(tour) - 1))
    for tour in ([0] + list(perm) + [0]
                 for perm in itertools.permutations(range(1, 4)))
)
print(f"  brute force over all permutations agrees: {tsp(distances) == brute}")
print("\n  The MASK is the DP state - 'which cities have I visited'. That")
print("  turns O(n!) into O(2^n * n^2), which is still exponential but")
print("  usable up to about 20 cities.")

## Interview questions

**1. What is Union-Find, and what is its complexity?**
> Disjoint sets with `find` and `union`. With path compression and union by rank, amortised O(α(n)) — effectively constant, since α(n) < 5 for any real n.

**2. Number of connected components / friend circles.**
> Union-Find, or BFS/DFS (**15.9**). Union-Find wins when edges arrive incrementally, because a traversal would have to re-run.

**3. Detect a cycle in an undirected graph.**
> `union` returning `False` means both endpoints were already connected — that edge closes a cycle.

**4. Kruskal's MST.** *(above)*
> Sort edges, union greedily, skip anything that would cycle. O(E log E). A good example of provably-correct greedy (**15.14**).

**5. Implement a trie with insert, search and startsWith.** *(above)*
> Dict of children plus an `is_word` flag. Stress that `is_word` is what separates a word from a prefix.

**6. Autocomplete / word search II.**
> Trie plus DFS. For word search on a grid, a trie lets you prune the moment a path stops being any word's prefix — far better than searching each word separately.

**7. Single number — every element appears twice except one.** *(above)*
> XOR everything. O(n) time, O(1) space.

**8. Count the set bits.** *(above)*
> Kernighan's `x &= x - 1`. Mention `int.bit_count()` exists since 3.10.

**9. Is x a power of two?** *(above)*
> `x > 0 and x & (x-1) == 0`. Explain the guard.

**10. Generate all subsets.**
> Bitmask counting 0 to 2ⁿ−1, or backtracking (**15.12**). Both O(2ⁿ).

**11. When would you use a trie over a hash map?**
> Only for prefix operations. Otherwise a `set` is faster and smaller — say so, it shows judgement.

**12. Travelling salesman for small n.** *(above)*
> Bitmask DP: O(2ⁿ·n²) rather than O(n!). Usable to roughly n = 20.

---

## Common Mistakes & Pitfalls

1. 🔴 **Union-Find without path compression or union by rank.** It degenerates into a chain and `find` becomes O(n).
2. 🔴 **A trie without an end-of-word flag.** You cannot distinguish a stored word from a prefix.
3. 🔴 **Assuming `~x` wraps like a fixed-width integer.** Python has no width: `~x == -x-1`. Mask with `& 0xFF` to emulate a byte.
4. 🔴 **Forgetting `x > 0` in the power-of-two test.** Zero would pass.
5. **Using a trie when you never query prefixes.** A `set` is faster and much smaller.
6. **XOR-swapping a variable with itself.** It becomes zero. Use tuple unpacking.
7. **Bitmask DP beyond ~20 items.** 2ⁿ states stops being tractable fast.
8. **Re-running BFS after every new edge** to track components. That is what Union-Find exists to avoid.
9. **Assuming Kruskal produced a spanning tree.** On a disconnected graph you get a forest with fewer than V−1 edges - check.

## Best Practices

- Implement Union-Find with both optimisations - they are four extra lines.
- Use `union`'s return value as a cycle detector rather than a separate check.
- Reach for a trie only when prefixes matter; otherwise a `set` (**15.6**).
- For a static word list, consider a sorted list plus `bisect` (**15.11**) before a trie.
- Use `int.bit_count()` in real code; know Kernighan's trick for interviews.
- Name your bit constants - `FLAG_ACTIVE = 1 << 2` beats a bare `4`.
- Use `format(x, 'b')` when debugging bit logic; the binary makes the bug obvious.
- Verify bit tricks against a readable implementation on random inputs.

## Practice Exercises

Try these before moving on.

1. Add a `rank` field to `UnionFind` instead of `size` and confirm the measured step counts are similar. Why do both work?
2. 🔴 Implement Union-Find *without* path compression but *with* union by size. How much of the benefit came from each optimisation?
3. Use Union-Find to solve 'number of islands' (**15.9**) and compare with the BFS version. Which is clearer, and which handles new land appearing?
4. Add `delete(word)` to the trie. What makes it harder than insertion - and when can you actually remove a node?
5. Implement 'word search II' using a trie to prune, and compare against searching each word independently.
6. Implement `single_number_ii`: every value appears three times except one. XOR alone no longer works - why, and what replaces it?
7. 🔴 Write `add(a, b)` using only bitwise operators - no `+`. Then explain why Python's arbitrary-precision negatives make this trickier than in C.
8. Extend the TSP solver to return the actual tour, not just its length (**15.13**'s reconstruction technique).